# Voice Agents

**Goal:** learn two practical ways to build a voice agent without making the code complicated.

1. **Sandwich method:** ElevenLabs STT -> Azure OpenAI with LangChain -> ElevenLabs TTS.
2. **Realtime method:** Azure OpenAI Realtime connection for low-latency interaction.

We will first build the text path, then add voice input, then add a small SPP layer, and finally compare it with the realtime pattern.

## 1. Setup

Run this install cell once if the packages are not available in your environment.

In [ ]:
# !pip install -q python-dotenv requests langchain-openai websockets streamlit ipywidgets lab-mic sounddevice

In [ ]:
import asyncio
import base64
import json
import os
import wave
from pathlib import Path

import requests
from dotenv import load_dotenv
from IPython.display import Audio, display
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import AzureChatOpenAI

load_dotenv('.env.example', override=True)

print('Environment loaded. Fill .env before running live API calls.')

## 2. Environment Variables

Use `.env.example` as the template. The checks below tell us what is still missing.

In [ ]:
required_for_sandwich = [
    'AZURE_OPENAI_API_KEY',
    'AZURE_OPENAI_ENDPOINT',
    'AZURE_OPENAI_API_VERSION',
    'AZURE_DEPLOYMENT',
    'ELEVENLABS_API_KEY',
    'ELEVENLABS_VOICE_ID',
]

required_for_realtime = [
    'AZURE_OPENAI_API_KEY',
    'AZURE_OPENAI_ENDPOINT',
    'AZURE_OPENAI_API_VERSION',
    'AZURE_REALTIME_DEPLOYMENT',
]

missing_sandwich = [name for name in required_for_sandwich if not os.getenv(name)]
missing_realtime = [name for name in required_for_realtime if not os.getenv(name)]

print('Missing for sandwich:', missing_sandwich)
print('Missing for realtime:', missing_realtime)

In [ ]:
import numpy as np
import sounddevice as sd
from scipy.io import wavfile
from IPython.display import Audio, display

# 1. Show available microphone devices
print("Available input devices:\n")

input_devices = []
for index, device in enumerate(sd.query_devices()):
    if device["max_input_channels"] > 0:
        input_devices.append(index)
        print(
            f"{index}: {device['name']} "
            f"| inputs={device['max_input_channels']} "
            f"| default_sr={device['default_samplerate']}"
        )

# 2. Select microphone device ID
device_id = int(input("\nEnter microphone device ID: "))

# 3. Record for 5 seconds
duration_seconds = 5
sample_rate = 16000
output_path = "mic-recording.wav"

print(f"\nRecording from device {device_id} for {duration_seconds} seconds...")

recording = sd.rec(
    int(duration_seconds * sample_rate),
    samplerate=sample_rate,
    channels=1,
    dtype="float32",
    device=device_id,
)

sd.wait()

# 4. Save as WAV
audio_int16 = np.clip(recording.squeeze() * 32767, -32768, 32767).astype(np.int16)
wavfile.write(output_path, sample_rate, audio_int16)

print(f"Recording saved to: {output_path}")

# 5. Play audio in notebook
display(Audio(output_path))


## 3. Text Input -> Azure OpenAI LangChain Response

Before voice, build the normal LLM path. This is the middle of the sandwich.

In [ ]:
os.environ['AZURE_OPENAI_API_KEY']

In [ ]:
llm = AzureChatOpenAI(
        azure_deployment=os.environ['AZURE_DEPLOYMENT'],
        api_version=os.getenv('AZURE_OPENAI_API_VERSION', '2025-04-01-preview'),
        temperature=0.2,
    )

llm.invoke('hello')


In [ ]:
def get_llm():
    return AzureChatOpenAI(
        azure_deployment=os.environ['AZURE_DEPLOYMENT'],
        api_version=os.getenv('AZURE_OPENAI_API_VERSION', '2025-04-01-preview'),
        temperature=0.2,
    )


def ask_llm(text: str) -> str:
    llm = get_llm()
    response = llm.invoke(
        [
            SystemMessage(
                content=(
                    'You are a helpful BFSI training assistant. '
                    'Answer clearly in 3 to 5 short sentences.'
                )
            ),
            HumanMessage(content=text),
        ]
    )
    return response.content


# Example after .env is filled:
ask_llm('Explain voice agents in simple terms.')

## 4. Voice Input -> ElevenLabs STT -> Azure OpenAI

Now we add speech-to-text. The output is just text, so the LLM function does not need to change.

In [ ]:
def speech_to_text(audio_path: str) -> str:
    url = 'https://api.elevenlabs.io/v1/speech-to-text'
    headers = {'xi-api-key': os.environ['ELEVENLABS_API_KEY']}
    data = {'model_id': os.getenv('ELEVENLABS_STT_MODEL', 'scribe_v2')}

    with open(audio_path, 'rb') as audio_file:
        files = {'file': (Path(audio_path).name, audio_file)}
        response = requests.post(url, headers=headers, data=data, files=files, timeout=120)

    response.raise_for_status()
    return response.json().get('text', '')


# Example after adding a short audio file:
transcript = speech_to_text('mic-recording.wav')
print(transcript)
print(ask_llm(transcript))

## 5. Add SPP: Speech Processing Pipeline

The SPP layer is intentionally small. It cleans the transcript and shapes it into a better LLM prompt.

In [ ]:
def apply_spp(transcript: str) -> str:
    cleaned = ' '.join(transcript.strip().split())
    return f'''
The user asked this by voice:
{cleaned}

Give a classroom-friendly answer. If the question is unclear, ask one short clarification question.
'''


# Example:
prompt = apply_spp('  explain    voice agents for banks  ')
print(prompt)
print(ask_llm(prompt))

## 6. Complete the Sandwich: Text-to-Speech

Now we turn the answer back into voice using ElevenLabs TTS.

In [ ]:
def text_to_speech(text: str, output_path: str) -> str:
    voice_id = os.environ['ELEVENLABS_VOICE_ID']
    model_id = os.getenv('ELEVENLABS_TTS_MODEL', 'eleven_multilingual_v2')
    url = f'https://api.elevenlabs.io/v1/text-to-speech/{voice_id}'

    response = requests.post(
        url,
        headers={
            'xi-api-key': os.environ['ELEVENLABS_API_KEY'],
            'Content-Type': 'application/json',
            'Accept': 'audio/mpeg',
        },
        json={'text': text, 'model_id': model_id},
        timeout=120,
    )
    response.raise_for_status()

    with open(output_path, 'wb') as audio_file:
        audio_file.write(response.content)
    return output_path


def sandwich_voice_agent(audio_path: str) -> dict:
    transcript = speech_to_text(audio_path)
    prompt = apply_spp(transcript)
    answer = ask_llm(prompt)
    spoken_answer = text_to_speech(answer, 'voice-agent-answer.mp3')
    return {
        'transcript': transcript,
        'prompt': prompt,
        'answer': answer,
        'audio_path': spoken_answer,
    }


# Example:
result = sandwich_voice_agent('mic-recording.wav')
print(result['transcript'])
print(result['answer'])
display(Audio(result['audio_path']))

In [ ]:
def stream_text_to_speech(text: str, output_path: str = 'voice-agent-streamed-answer.mp3') -> str:
    voice_id = os.environ['ELEVENLABS_VOICE_ID']
    model_id = os.getenv('ELEVENLABS_TTS_MODEL', 'eleven_multilingual_v2')
    url = f'https://api.elevenlabs.io/v1/text-to-speech/{voice_id}/stream'

    with requests.post(
        url,
        headers={
            'xi-api-key': os.environ['ELEVENLABS_API_KEY'],
            'Content-Type': 'application/json',
            'Accept': 'audio/mpeg',
        },
        json={'text': text, 'model_id': model_id},
        stream=True,
        timeout=120,
    ) as response:
        response.raise_for_status()
        with open(output_path, 'wb') as audio_file:
            for chunk in response.iter_content(chunk_size=4096):
                if chunk:
                    audio_file.write(chunk)

    return output_path


# Example:
streamed_path = stream_text_to_speech('This audio was generated with the streaming endpoint.')
display(Audio(streamed_path))

## 7. Realtime Method

The realtime pattern removes the separate STT and TTS services. The model connection can accept audio and return responses over a realtime channel.

For classroom simplicity, this helper shows a small server-side WebSocket flow. If audio is used, provide a short mono PCM16 WAV file at 16 kHz or 24 kHz.

In [ ]:
def assert_realtime_wav(audio_path: str) -> None:
    with wave.open(audio_path, 'rb') as wav_file:
        channels = wav_file.getnchannels()
        sample_width = wav_file.getsampwidth()
        frame_rate = wav_file.getframerate()

    if channels != 1 or sample_width != 2 or frame_rate not in (16000, 24000):
        raise ValueError('Use mono PCM16 WAV at 16 kHz or 24 kHz for this simple realtime demo.')


async def connect_realtime(websockets, url: str, headers: dict):
    try:
        return await websockets.connect(url, additional_headers=headers)
    except TypeError:
        return await websockets.connect(url, extra_headers=headers)


async def realtime_response(prompt: str, audio_path: str | None = None) -> dict:
    import websockets

    endpoint = os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')
    deployment = os.environ['AZURE_REALTIME_DEPLOYMENT']
    api_version = os.getenv('AZURE_OPENAI_API_VERSION', '2025-04-01-preview')
    resource_host = endpoint.replace('https://', '').replace('http://', '')
    url = f'wss://{resource_host}/openai/realtime?api-version={api_version}&deployment={deployment}'

    transcript_parts = []
    answer_parts = []

    headers = {'api-key': os.environ['AZURE_OPENAI_API_KEY']}
    ws = await connect_realtime(websockets, url, headers)
    try:
        await ws.send(json.dumps({
            'type': 'session.update',
            'session': {
                'modalities': ['text'],
                'input_audio_format': 'pcm16',
                'input_audio_transcription': {'model': 'whisper-1'},
                'instructions': 'You are a concise BFSI voice agent demo assistant.',
            },
        }))

        if audio_path:
            assert_realtime_wav(audio_path)
            with open(audio_path, 'rb') as audio_file:
                encoded_audio = base64.b64encode(audio_file.read()).decode('utf-8')
            await ws.send(json.dumps({'type': 'input_audio_buffer.append', 'audio': encoded_audio}))
            await ws.send(json.dumps({'type': 'input_audio_buffer.commit'}))

        await ws.send(json.dumps({
            'type': 'response.create',
            'response': {
                'modalities': ['text'],
                'instructions': prompt,
            },
        }))

        while True:
            event = json.loads(await ws.recv())
            event_type = event.get('type', '')

            if event_type in ('response.text.delta', 'response.output_text.delta'):
                answer_parts.append(event.get('delta', ''))
            elif event_type == 'conversation.item.input_audio_transcription.completed':
                transcript_parts.append(event.get('transcript', ''))
            elif event_type == 'response.done':
                break
            elif event_type == 'error':
                raise RuntimeError(event.get('error', {}).get('message', event))
    finally:
        await ws.close()

    return {
        'transcript': ' '.join(transcript_parts).strip(),
        'answer': ''.join(answer_parts).strip(),
    }


# Text-only realtime smoke test:
result = await realtime_response('Explain realtime voice agents in one paragraph.')
print(result['answer'])

## 8. Live Notebook Audio Capture and Playback

There are two notebook-friendly ways to capture audio:

1. **Native kernel recorder with `sounddevice`** - recommended when teaching on Windows or when browser permissions block microphone access.
2. **Browser widget recorder with `lab-mic`** - useful in JupyterLab when the browser allows microphone access.

If the widget shows `Microphone Error: NotAllowedError`, the browser has denied microphone access. Use the native recorder below, or reset the browser permission for the Jupyter URL and reload the notebook.

The full live flow is:

1. Record a short question as `live-question.wav`.
2. Play it in the notebook.
3. Process it through STT -> SPP -> LLM.
4. Generate streamed TTS and play the output in the notebook.

In [ ]:
def run_live_notebook_sandwich(audio_path: str = 'live-question.wav') -> dict:
    transcript = speech_to_text(audio_path)
    prompt = apply_spp(transcript)
    answer = ask_llm(prompt)
    audio_output = stream_text_to_speech(answer, 'live-streamed-answer.mp3')

    print('Transcript:')
    print(transcript)
    print()
    print('Answer:')
    print(answer)
    display(Audio(audio_output))

    return {
        'transcript': transcript,
        'prompt': prompt,
        'answer': answer,
        'audio_path': audio_output,
    }


# Full live notebook flow after recording:
live_result = run_live_notebook_sandwich('mic-recording.wav')

## 9. Streamlit App

After the notebook concepts are clear, run the app:

```powershell
streamlit run "8. Voice Agents/app.py"
```

Use the app to compare the sandwich method with the realtime method.